In [ ]:
#!/usr/bin/env python3
"""
Task & Project Management System
A fully modular, Object-Oriented CLI Application built with standard Python libraries.
"""

import json
import os
from datetime import datetime
from enum import Enum
from typing import List, Optional, Dict, Any


# ==========================================
# 1. ENUMS AND CUSTOM EXCEPTIONS
# ==========================================

class Status(str, Enum):
    TODO = "To Do"
    IN_PROGRESS = "In Progress"
    COMPLETED = "Completed"


class Priority(str, Enum):
    LOW = "Low"
    MEDIUM = "Medium"
    HIGH = "High"


class TaskNotFoundError(Exception):
    """Raised when a task ID is not found."""
    pass


class ProjectNotFoundError(Exception):
    """Raised when a project is not found."""
    pass


# ==========================================
# 2. DATA MODELS
# ==========================================

class Task:
    """Represents an individual task within a project."""

    def __init__(self, task_id: int, title: str, description: str = "", 
                 priority: Priority = Priority.MEDIUM):
        self.id = task_id
        self.title = title
        self.description = description
        self.status = Status.TODO
        self.priority = priority
        self.created_at = datetime.now().strftime("%Y-%m-%d %H:%M")

    def mark_completed(self) -> None:
        self.status = Status.COMPLETED

    def update_status(self, new_status: Status) -> None:
        self.status = new_status

    def to_dict(self) -> Dict[str, Any]:
        """Convert object attributes to a dictionary for JSON serialization."""
        return {
            "id": self.id,
            "title": self.title,
            "description": self.description,
            "status": self.status.value,
            "priority": self.priority.value,
            "created_at": self.created_at,
        }

    @classmethod
    def from_dict(cls, data: Dict[str, Any]) -> 'Task':
        """Reconstruct a Task object from a dictionary."""
        task = cls(
            task_id=data["id"],
            title=data["title"],
            description=data["description"],
            priority=Priority(data["priority"])
        )
        task.status = Status(data["status"])
        task.created_at = data["created_at"]
        return task

    def __str__(self) -> str:
        return f"[{self.id}] {self.title} | Status: {self.status.value} | Priority: {self.priority.value}"


class Project:
    """Represents a project containing multiple tasks."""

    def __init__(self, name: str, description: str = ""):
        self.name = name
        self.description = description
        self.tasks: List[Task] = []
        self._next_task_id = 1

    def add_task(self, title: str, description: str, priority: Priority) -> Task:
        task = Task(self._next_task_id, title, description, priority)
        self.tasks.append(task)
        self._next_task_id += 1
        return task

    def get_task(self, task_id: int) -> Task:
        for task in self.tasks:
            if task.id == task_id:
                return task
        raise TaskNotFoundError(f"Task with ID {task_id} not found in project '{self.name}'.")

    def delete_task(self, task_id: int) -> bool:
        task = self.get_task(task_id)
        self.tasks.remove(task)
        return True

    def to_dict(self) -> Dict[str, Any]:
        return {
            "name": self.name,
            "description": self.description,
            "next_task_id": self._next_task_id,
            "tasks": [task.to_dict() for task in self.tasks]
        }

    @classmethod
    def from_dict(cls, data: Dict[str, Any]) -> 'Project':
        project = cls(name=data["name"], description=data["description"])
        project._next_task_id = data["next_task_id"]
        project.tasks = [Task.from_dict(t) for t in data["tasks"]]
        return project


# ==========================================
# 3. MANAGER & STORAGE ENGINE
# ==========================================

class ProjectManager:
    """Manages collection of projects and persistent storage."""

    def __init__(self, storage_file: str = "projects_data.json"):
        self.storage_file = storage_file
        self.projects: Dict[str, Project] = {}
        self.load_data()

    def create_project(self, name: str, description: str = "") -> Project:
        if name in self.projects:
            raise ValueError(f"Project '{name}' already exists.")
        project = Project(name, description)
        self.projects[name] = project
        self.save_data()
        return project

    def get_project(self, name: str) -> Project:
        if name not in self.projects:
            raise ProjectNotFoundError(f"Project '{name}' does not exist.")
        return self.projects[name]

    def save_data(self) -> None:
        """Saves current state to JSON file."""
        data = {name: proj.to_dict() for name, proj in self.projects.items()}
        with open(self.storage_file, "w") as f:
            json.dump(data, f, indent=4)

    def load_data(self) -> None:
        """Loads state from JSON file if available."""
        if not os.path.exists(self.storage_file):
            return
        
        try:
            with open(self.storage_file, "r") as f:
                data = json.load(f)
                for proj_name, proj_data in data.items():
                    self.projects[proj_name] = Project.from_dict(proj_data)
        except (json.JSONDecodeError, KeyError) as e:
            print(f"[Warning] Failed to parse store data cleanly: {e}")


# ==========================================
# 4. COMMAND LINE INTERFACE
# ==========================================

class CLI:
    def __init__(self):
        self.manager = ProjectManager()

    def run(self):
        while True:
            print("\n" + "="*40)
            print("   PROJECT & TASK MANAGER CLI   ")
            print("="*40)
            print("1. List Projects")
            print("2. Create Project")
            print("3. View/Manage Project Tasks")
            print("4. Add Task to Project")
            print("5. Update Task Status")
            print("6. Delete Task")
            print("7. Exit")
            
            choice = input("\nSelect an option (1-7): ").strip()
            print()

            try:
                if choice == "1":
                    self.list_projects()
                elif choice == "2":
                    self.create_project()
                elif choice == "3":
                    self.view_project_tasks()
                elif choice == "4":
                    self.add_task()
                elif choice == "5":
                    self.update_task_status()
                elif choice == "6":
                    self.delete_task()
                elif choice == "7":
                    print("Saving and exiting... Goodbye!")
                    break
                else:
                    print("Invalid choice. Please select 1-7.")
            except Exception as e:
                print(f"Error: {e}")

    def list_projects(self):
        if not self.manager.projects:
            print("No projects available.")
            return
        print("Projects List:")
        for name, proj in self.manager.projects.items():
            print(f" • {name} - Tasks: {len(proj.tasks)} ({proj.description})")

    def create_project(self):
        name = input("Enter project name: ").strip()
        desc = input("Enter project description: ").strip()
        self.manager.create_project(name, desc)
        print(f"Project '{name}' created successfully.")

    def view_project_tasks(self):
        proj_name = input("Enter project name: ").strip()
        project = self.manager.get_project(proj_name)
        
        print(f"\n--- Tasks for {project.name} ---")
        if not project.tasks:
            print("No tasks in this project.")
            return
        for task in project.tasks:
            print(task)

    def add_task(self):
        proj_name = input("Enter project name: ").strip()
        project = self.manager.get_project(proj_name)
        
        title = input("Task Title: ").strip()
        desc = input("Task Description: ").strip()
        print("Priority levels: 1. Low, 2. Medium, 3. High")
        p_choice = input("Select Priority (default 2): ").strip()
        
        priority_map = {"1": Priority.LOW, "2": Priority.MEDIUM, "3": Priority.HIGH}
        priority = priority_map.get(p_choice, Priority.MEDIUM)

        task = project.add_task(title, desc, priority)
        self.manager.save_data()
        print(f"Task #{task.id} '{title}' added successfully!")

    def update_task_status(self):
        proj_name = input("Enter project name: ").strip()
        project = self.manager.get_project(proj_name)
        
        task_id = int(input("Enter Task ID: "))
        task = project.get_task(task_id)

        print("Select new status:")
        print("1. To Do\n2. In Progress\n3. Completed")
        st_choice = input("Choice: ").strip()

        status_map = {"1": Status.TODO, "2": Status.IN_PROGRESS, "3": Status.COMPLETED}
        if st_choice in status_map:
            task.update_status(status_map[st_choice])
            self.manager.save_data()
            print(f"Task #{task.id} updated to '{task.status.value}'.")
        else:
            print("Invalid status option.")

    def delete_task(self):
        proj_name = input("Enter project name: ").strip()
        project = self.manager.get_project(proj_name)
        
        task_id = int(input("Enter Task ID to delete: "))
        if project.delete_task(task_id):
            self.manager.save_data()
            print(f"Task #{task_id} deleted successfully.")


if __name__ == "__main__":
    app = CLI()
    app.run()
